# 0.3 Your first LLM call: raw HTTP vs LangChain

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

## Learning objectives

- Make the same LLM call three ways: raw HTTPS, the official provider SDK, and LangChain.
- Explain what each layer adds and what it hides.
- Read token usage from a LangChain response.
- Decide when to use LangChain and when to skip it.

## The idea in plain words

An LLM API is, at bottom, an HTTPS endpoint. You POST some JSON, you get some JSON back. Everything else, SDKs and frameworks included, is convenience layered on top of that. This lesson builds the same call three times so you can feel exactly what each layer buys you and what it costs. When something breaks later in the course, you will debug it at the right layer instead of guessing.

We will ask the model one question: "In one sentence, what is an agent?" The prompt stays identical. Only the plumbing changes.

## Layer 1: raw HTTPS

No libraries except the standard library. This is what is actually happening on the wire:

In [ ]:
import json
import os
import urllib.request

api_key = os.environ["OPENAI_API_KEY"]  # set in .env (Lesson 0.2)

payload = {
    "model": "gpt-4o-mini",
    "messages": [{"role": "user", "content": "In one sentence, what is an agent?"}],
}

request = urllib.request.Request(
    "https://api.openai.com/v1/chat/completions",
    data=json.dumps(payload).encode(),
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
)

with urllib.request.urlopen(request, timeout=60) as response:
    body = json.loads(response.read())

print(body["choices"][0]["message"]["content"])

It works. It is also tedious: you build headers by hand, parse nested JSON by hand, and there are no retries. If the network hiccups or the API rate-limits you, this script just crashes. Every provider has a slightly different URL shape and response shape, so switching providers means rewriting this.

## Layer 2: the official provider SDK

The `openai` package (version 2.x, pulled in automatically by `langchain-openai`) wraps the HTTP layer. Note that the SDK's primary API is now the Responses API; Chat Completions is still fully supported:

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()  # reads OPENAI_API_KEY from the environment

# Primary API in SDK v2:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions="Answer in one sentence.",
    input="What is an agent?",
)
print(response.output_text)

# The older Chat Completions API still works and is supported:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "In one sentence, what is an agent?"}],
)
print(completion.choices[0].message.content)

What you gained: typed objects instead of raw dicts, proper error classes instead of HTTP status codes, and both sync and async clients. What you did not gain: this SDK only talks to OpenAI (or OpenAI-compatible endpoints). Swap to another provider and you learn a new SDK.

## Layer 3: LangChain

Same question, through LangChain's chat model abstraction:

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")

# A plain string becomes a human message under the hood:
answer = llm.invoke("In one sentence, what is an agent?")
print(answer.content)
print(answer.usage_metadata)  # {'input_tokens': ..., 'output_tokens': ..., 'total_tokens': ...}

Three lines of real logic. Now the part that matters for the rest of the course: swap the provider and nothing else changes.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b")  # local, free, private
answer = llm.invoke("In one sentence, what is an agent?")
print(answer.content)

The `.invoke()` interface, the returned message object, and `usage_metadata` are identical. Your chains, agents, and graphs in later modules are written against this interface, not against any one provider. That is the core thing LangChain buys you: **a stable seam between your code and whoever serves the model**.

## So what does the framework actually buy you?

Be concrete about it, because "it makes things easier" is not an engineering reason:

1. **Provider swapping.** `ChatOpenAI` and `ChatOllama` share one interface. Moving from a hosted model to a local one (or between hosted providers) is a constructor change.
2. **Retries and timeouts.** Transient failures (rate limits, network blips) are retried with backoff instead of crashing your script. Configurable via `max_retries` on the model.
3. **Token accounting.** `usage_metadata` on every response gives you input/output token counts in one place, which is how you will track cost in Module 6.
4. **Composition.** The object returned by `.invoke()` plugs directly into prompts, parsers, tools, and agents in later modules. Raw dicts do not.
5. **Observability hooks.** Later, when you add LangSmith tracing, it attaches to these same objects with no code changes.

And what it costs you: the HTTP layer is now hidden, so provider-specific errors arrive wrapped in framework exceptions and take longer to diagnose. You inherit a dependency tree that changes weekly (hence pinned versions). For a one-off script that calls one API, raw HTTPS or the provider SDK is the right call. LangChain earns its place when you start composing: prompts plus models plus tools plus memory, which is exactly what Modules 1 and 2 do.

## What can go wrong

- **401 Unauthorized.** Your key is missing, wrong, or not loaded. Check that `.env` is in the folder you run from and that `load_dotenv()` is called. In Layer 1, the same failure is an HTTPError with status 401 and no helpful message.
- **429 rate limit.** You called too fast or the account hit its limit. Layers 2 and 3 retry automatically a few times; Layer 1 crashes immediately. If 429s persist, slow down or check your account's limits and spending cap.
- **Model name typo.** `gpt-4o-mini` vs `gpt-4o_mini` gives you a 404-style "model not found" error that looks scarier than it is. Copy model names exactly.
- **Responses vs Chat Completions confusion.** SDK v2 examples use `client.responses.create(...)` with `input=`, while older tutorials use `client.chat.completions.create(...)` with `messages=`. Both work. Do not mix their argument shapes: `input=` belongs to Responses, `messages=` belongs to Chat Completions.
- **Ollama connection refused.** `ChatOllama` fails if the Ollama server is not running. Start it with `ollama serve` and confirm the model is pulled (`ollama list`).
- **Silent cost.** A script in a loop calling a hosted model spends real money per token. During learning, prefer the local Ollama path for experiments and save hosted calls for when you need the stronger model.

## Check your understanding

**1. What is actually happening on the wire when you call an LLM API?**
An HTTPS POST of a JSON payload (model name, messages, parameters) to the provider's endpoint, with the API key in the Authorization header; the response is JSON containing the generated text plus metadata.

**2. What is the single biggest thing LangChain's chat model abstraction buys you over the provider SDK?**
A provider-independent interface: `ChatOpenAI`, `ChatOllama`, and others share `.invoke()` and the same message objects, so swapping providers is a constructor change rather than a rewrite.

**3. Name two things the provider SDK gives you that raw HTTPS does not.**
Any two of: typed request/response objects instead of raw dicts; proper error classes; built-in sync and async clients; the newer Responses API with `output_text`.

**4. Your script gets a 429 error. How does each layer behave?**
Raw HTTPS (Layer 1) crashes unless you wrote retry logic yourself. The provider SDK and LangChain retry transient failures with backoff automatically (LangChain's retry count is configurable via `max_retries`).

**5. When would you NOT use LangChain for an LLM call?**
For a one-off script that calls a single provider's API: raw HTTPS or the provider SDK has fewer dependencies, less version churn, and errors arrive unwrapped and easier to diagnose.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.